# 03 — Propulsion Sizing

Pick a motor / prop / ESC combination that meets the T/W requirement
from notebook 01 at the AUW from notebook 02.


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "multirotor"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "multirotor"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(PLATFORM_DIR / "design_point.yaml") as f:
    design = yaml.safe_load(f)
with open(PLATFORM_DIR / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

# Inputs (override here if running standalone)
auw_kg = design["design_point"]["target_auw_kg"]
tw = design["design_point"]["thrust_to_weight_ratio"]
hover_throttle = design["design_point"]["hover_throttle_fraction"]
n_motors = airframe["airframe"]["num_motors"]
pack_voltage_nominal_v = 22.2  # 6S
pack_voltage_full_v = 25.2
prop_diameter_in = 18


## Required thrust per motor


In [ ]:
from uavdesign import propulsion as prop

T_peak_per_motor_n = prop.thrust_per_motor(
    auw_kg, n_motors, tw * design["design_point"]["safety_factor"]
)
T_hover_per_motor_n = prop.hover_thrust_per_motor(auw_kg, n_motors)

print(f"Peak thrust per motor (with SF):  {T_peak_per_motor_n:.2f} N "
      f"({T_peak_per_motor_n/prop.G*1000:.0f} gf)")
print(f"Hover thrust per motor:           {T_hover_per_motor_n:.2f} N "
      f"({T_hover_per_motor_n/prop.G*1000:.0f} gf)")
print(f"Hover throttle target fraction:   {hover_throttle:.0%}")


## KV target

Rule of thumb: KV ≈ unloaded RPM target / pack voltage. Real loaded
RPM sags 15–25 %.


In [ ]:
# Target tip Mach ~0.5 for low noise; back-solve unloaded RPM target.
prop_d_m = prop_diameter_in * 0.0254
target_tip_speed_m_s = 170.0  # ~Mach 0.5 at sea level
rpm_target = (target_tip_speed_m_s * 60.0) / (np.pi * prop_d_m)
kv = prop.motor_kv_estimate(pack_voltage_full_v, rpm_target)

print(f"Target tip speed:        {target_tip_speed_m_s:.0f} m/s")
print(f"Unloaded RPM target:     {rpm_target:.0f} RPM")
print(f"Indicative motor KV:     {kv:.0f} rpm/V (at {pack_voltage_full_v} V full)")


## Candidate motor/prop/ESC shortlist

Fill in datasheet data (thrust, current, efficiency at the chosen prop)
to compare. Below is a minimal placeholder table; replace values from
manufacturer test reports.


In [ ]:
import pandas as pd

candidates = pd.DataFrame([
    {"motor": "Tarot 5008 340 KV", "prop": "1855 CF",
     "hover_thrust_g": 1100, "hover_current_a": 8.0, "peak_current_a": 28.0,
     "efficiency_g_per_w": 8.5},
    {"motor": "T-MOTOR MN505-S 320 KV", "prop": "1855 CF",
     "hover_thrust_g": 1180, "hover_current_a": 7.5, "peak_current_a": 32.0,
     "efficiency_g_per_w": 9.2},
    {"motor": "Sunnysky X4108S 380 KV", "prop": "1755 CF",
     "hover_thrust_g": 980,  "hover_current_a": 8.8, "peak_current_a": 26.0,
     "efficiency_g_per_w": 7.8},
])

T_hover_per_motor_g = T_hover_per_motor_n / prop.G * 1000
candidates["hover_margin_g"] = candidates["hover_thrust_g"] - T_hover_per_motor_g
candidates["esc_min_a_20pct_headroom"] = (candidates["peak_current_a"] * 1.20).round(1)
candidates


## Hover power estimate (momentum theory)


In [ ]:
rho = prop.isa_density(mission["mission"]["altitude_asl_m"])
A_disc_per_motor = prop.disc_area_m2(prop_diameter_in, 1)
P_hover_per_motor_w = prop.prop_power_estimate(T_hover_per_motor_n, A_disc_per_motor, rho)
P_hover_total_w = P_hover_per_motor_w * n_motors
I_hover_pack_a = P_hover_total_w / pack_voltage_nominal_v

print(f"Hover shaft power per motor: {P_hover_per_motor_w:.0f} W")
print(f"Hover shaft power total:     {P_hover_total_w:.0f} W")
print(f"Implied pack current @ {pack_voltage_nominal_v} V: {I_hover_pack_a:.1f} A")
print("(Multiply by ~1/eta_motor_esc ~ 0.85 for electrical pack current.)")


## Hand-off to power + endurance notebooks


In [ ]:
propulsion_summary = {
    "n_motors": n_motors,
    "prop_diameter_in": prop_diameter_in,
    "kv_indicative": float(kv),
    "hover_thrust_per_motor_n": T_hover_per_motor_n,
    "peak_thrust_per_motor_n": T_peak_per_motor_n,
    "hover_shaft_power_total_w": P_hover_total_w,
    "hover_pack_current_a": I_hover_pack_a,
    "pack_voltage_nominal_v": pack_voltage_nominal_v,
}
propulsion_summary
